<a href="https://colab.research.google.com/github/gibsonx/jlpt_simulator/blob/dev/graphs/n3/outliner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/gibsonx/jlpt_simulator.git
    %cd jlpt_simulator
    !git checkout dev
    !apt-get install python3-dev graphviz libgraphviz-dev pkg-config
    !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [2]:
import json
import logging
import random
import time
import pandas as pd
import yaml
import inspect
from tqdm import tqdm
import os
from libs.Logger import logger
from datetime import datetime
from docx import Document
from html4docx import HtmlToDocx
import uuid
from libs.CosmosMongoDB import CosmosMongoDB
from libs.LLMs import *
from IPython.display import display, Markdown, HTML
import datetime
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from libs.Utils import render_to_html
load_dotenv()

from graphs.common.TaskRunner import TaskRunner

# N1 Level Exam

In [3]:
runner = TaskRunner(level="N1", exam_type="full_exam")
n1_outline, n1_exam_paper = runner.run()

2025-11-17 13:21:39,181 - INFO - jlpt - Module 'graphs.n1.outliner' imported successfully.
2025-11-17 13:21:39,181 - INFO - Module 'graphs.n1.outliner' imported successfully.
2025-11-17 13:21:39,183 - INFO - jlpt - Current Task ID None is being initialized
2025-11-17 13:21:39,183 - INFO - Current Task ID None is being initialized
2025-11-17 13:22:04,947 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-17 13:22:04,966 - INFO - jlpt - Outline of the exam:

# 日本語能力試験N1 模擬試験問題

## 第1部：語彙

### kanji_reading

6問：名詞2問、動詞2問、形容詞1問、副詞1問。80%は非常に難しい語彙を選定。すべて漢字表記。

- **概念**
- **定義**
- **従う**
- **携わる**
- **脆い**
- **逐一**

### word_meaning

7問：名詞2問、動詞2問、形容詞2問、副詞1問。80%は非常に難しい語彙を選定。語彙はひらがな表記。

- **きょうくん**
- **しゅうし**
- **みあわせる**
- **おさまる**
- **ばかばかしい**
- **しっそ**
- **きっかり**

### synonym_substitution

6問：名詞2問、動詞3問、形容詞1問。80%は非常に難しい語彙を選定。語彙はひらがな表記。

- **じそんしん**
- **のうにゅう**

## N1 Outline Preview

In [4]:
display(Markdown(n1_outline.as_str))

# 日本語能力試験N1 模擬試験問題

## 第1部：語彙

### kanji_reading

6問：名詞2問、動詞2問、形容詞1問、副詞1問。80%は非常に難しい語彙を選定。すべて漢字表記。

- **概念**
- **定義**
- **従う**
- **携わる**
- **脆い**
- **逐一**

### word_meaning

7問：名詞2問、動詞2問、形容詞2問、副詞1問。80%は非常に難しい語彙を選定。語彙はひらがな表記。

- **きょうくん**
- **しゅうし**
- **みあわせる**
- **おさまる**
- **ばかばかしい**
- **しっそ**
- **きっかり**

### synonym_substitution

6問：名詞2問、動詞3問、形容詞1問。80%は非常に難しい語彙を選定。語彙はひらがな表記。

- **じそんしん**
- **のうにゅう**
- **すます**
- **まじえる**
- **まかす**
- **なれなれしい**

### word_usage

6問：名詞3問、動詞2問、その他1問。80%は非常に難しい語彙を選定。名詞・動詞は漢字表記。

- **弁護**
- **概説**
- **領収書**
- **損なう**
- **携わる**
- **まんまるい**

## 第2部：文法

### sentence_grammar

10問：敬語1問、副詞1問、助詞1問、その他10種類の異なる文型。トピックはランダムに選択。

- **店で価格を尋ねる**...おいくらでございますか（敬語）
- **交通状況について話す**...いかにも
- **家族について話す**...に限ったことではない
- **健康とフィットネスについて話す**...までもない
- **時事問題について話す**...いかんによらず
- **教育について話す**...に即して
- **将来の抱負について話す**...べく
- **課題と解決策について話す**...を踏まえて
- **地域社会への貢献について話す**...ならでは
- **異文化交流について話す**...を機に

### sentence_sort

5問：文の並び替え問題。トピックはランダムに選択。

- **購入したい商品の説明**...ものと思われる
- **レストランで食べ物を注文する**...ことこの上ない
- **旅行の計画について話す**...に至るまで
- **ペットについて話す**...まみれ
- **家の改善について話す**...を兼ねて

### sentence_structure

1問：短い文章を読んで、4つの異なる文法ポイントを統合した問題。

- **技術について話す**...に至る／...に即して／...べく／...を踏まえて

## 第3部：読解

### short_passage_narrative_read

1問：短い物語文を読んで設問に答える。

- **ショッピング体験を説明する**

### short_passage_mail_read

1問：短いメール文を読んで設問に答える。

- **友人との日常会話**

### short_passage_narrative_read

1問：短い物語文を読んで設問に答える。

- **健康診断や医者への訪問について話す**

### short_passage_notification_read

1問：通知文を読んで設問に答える。

- **公共施設の利用方法について話す**

### midsize_passage_read

4問：中程度の長さの文章を読んで設問に答える。

- **最近の映画について話す**
- **キャリア目標について話す**
- **家事の分担について話す**
- **海外旅行の体験を話す**

### long_passage_read

1問：長文を読んで設問に答える。

- **環境問題について話す**

### understanding_read

1問：相談文と回答文を読んで設問に答える。

- **言語学習のコツについて話す**

### long_passage_read

1問：長文を読んで設問に答える。

- **芸術と文化について話す**

### info_retrieval

1問：情報検索問題。

- **地元の観光名所について話す**

## 第4部：聴解

### topic_understanding_txt

5問：質問を聞いてから話を聞き、最もよい選択肢を選ぶ。

- **おすすめを尋ねる**
- **週末の予定について話す**
- **ファッションとスタイルについて話す**
- **引っ越しの準備について話す**
- **好きな季節について話す**

### keypoint_understanding

6問：質問を聞き、問題用紙を見てから話を聞き、最もよい選択肢を選ぶ。

- **支払い方法について話す**
- **領収書を求める**
- **趣味について話す**
- **スポーツについて話す**
- **新しいスキルを学ぶ計画について話す**
- **特別な日の計画について話す**

### summary_understanding

5問：話を聞いてから質問と選択肢を聞き、最もよい選択肢を選ぶ。

- **バスの時刻表を尋ねる**
- **通勤について説明する**
- **天気の状況について話す**
- **本について話す**
- **ガーデニングについて話す**

### immediate_ack

11問：文を聞いてから返事を聞き、最もよい選択肢を選ぶ。

- **料理を褒める**
- **交通手段について話す**
- **タクシーを予約する**
- **電車の切符を買う**
- **道を尋ねる**
- **食事の好みについて話す**
- **家族について話す**
- **旅行の計画について話す**
- **最近のニュースについて意見を述べる**
- **ペットについて話す**
- **家の改善について話す**

### comprehensive_expression_listen_answer

1問：長めの話を聞いて設問に答える。

- **日本の祭りや文化イベントについて話す**

### comprehensive_expression_show_answer

2問：長めの話を聞いて設問に答える。

- **地域のイベントについて話す**
- **引退後の生活について話す**

## N1 HTML Result

In [5]:
html_output = render_to_html(n1_exam_paper['sections'])
display(HTML(html_output))
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N1_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N2 Level Exam

In [6]:
runner = TaskRunner(level="N2", exam_type="fast_exam")
n2_outline, n2_exam_paper = runner.run()

2025-11-17 13:48:41,410 - INFO - jlpt - Module 'graphs.n2.outliner' imported successfully.
2025-11-17 13:48:41,410 - INFO - Module 'graphs.n2.outliner' imported successfully.
2025-11-17 13:48:41,411 - INFO - jlpt - Current Task ID None is being initialized
2025-11-17 13:48:41,411 - INFO - Current Task ID None is being initialized
2025-11-17 13:48:54,081 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-17 13:48:54,087 - INFO - jlpt - Outline of the exam:

# 日本語能力試験N2 模擬試験問題

## 語彙（Vocabulary）

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい。名詞1問。80%難易度高い語彙。

- **しへい**

### write_kanji

問題2：このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい。名詞1問。

- **にょうぼう**

### words_collocation

問題3：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい。動詞1問。

- **ひきうける**

### word_meaning

問題4：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい。動詞1問。

- **あきれる**

### synonym_substitution

問題5：意味が最も近い

## N2 Outline Preview

In [7]:
display(Markdown(n2_outline.as_str))

# 日本語能力試験N2 模擬試験問題

## 語彙（Vocabulary）

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい。名詞1問。80%難易度高い語彙。

- **しへい**

### write_kanji

問題2：このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい。名詞1問。

- **にょうぼう**

### words_collocation

問題3：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい。動詞1問。

- **ひきうける**

### word_meaning

問題4：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい。動詞1問。

- **あきれる**

### synonym_substitution

問題5：意味が最も近いものを、1・2・3・4から一つえらびなさい。動詞1問。

- **こらえる**

### word_usage

問題6：つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。動詞1問。

- **ちぎる**

## 文法（Grammar）

### sentence_grammar

問題1：つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい。副詞1問。

- **週末の予定について話す**いよいよ（いよいよ）

### sentence_sort

問題2：つぎの文の ★ に入る最もよいものを、1・2・3・4から一つえらびなさい。2問。

- **健康診断や医者への訪問について話す**に際して（にさいして）
- **家事の分担について話す**つつ（つつ）

### sentence_structure

問題3：つぎの文章を読んで、文章全体の内容を考えて、文中の 48 から 51 の中に入る最もよいものを、1・2・3・4から一つえらびなさい。1問。

- **技術について話す**に基づいて（にもとづいて）

## 読解（Reading Comprehension）

### short_passage_narrative_read

問題1-1：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **最近の映画について話す**

### short_passage_mail_read

問題1-2：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **友人へのプレゼント選びについて話す**

### short_passage_notification_read

問題1-3：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **公共施設の利用方法について話す**

### midsize_passage_read

問題2：つぎの(1)と(2)の文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **環境問題について話す**

### comprehensive_read

問題3：つぎの(1)と(2)の文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **異文化交流について話す**

### long_passage_read

問題4：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。1記事。

- **キャリア目標について話す**

### info_retrieval

問題5：これを読んで、下の質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。情報検索。1記事。

- **日本の祭りや文化イベントについて話す**

## 聴解（Listening Comprehension）

### topic_understanding

問題1：まず質問を聞いてください。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。1問。

- **レストランで食べ物を注文する**

### keypoint_understanding

問題2：まず質問を聞いてください。そのあと、問題用紙を見てください。読む時間があります。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。1問。

- **購入したい商品の説明**

### summary_understanding

問題3：問題用紙に何も印刷されていません。この問題は、全体としてどんな内容かを聞く問題です。話の前に質問はありません。まず話を聞いてください。それから、質問と選択肢を聞いて、1から4の中から、最もよいものを一つえらんでください。1問。

- **道を尋ねる**

### immediate_ack

問題4：問題用紙に何も印刷していません。まず文を聞いてください。それから、その返事を聞いて、1から3の中から、最もよいものを一つえらんでください。1問。

- **交通手段について話す**

### comprehensive_expression_listen_answer

問題5-1：長めの話を聞きます。この問題には練習はありません。問題用紙にメモをとってもかまいません。1問。

- **支払い方法について話す**

### comprehensive_expression_show_answer

問題5-2：長めの話を聞きます。この問題には練習はありません。問題用紙にメモをとってもかまいません。1問。

- **割引交渉**

## N2 Exam Result

In [8]:
html_output = render_to_html(n2_exam_paper['sections'])
display(HTML(html_output))

名称,場所,開催日,内容,参加方法,料金
桜灯り祭,長野県松本市,4月6日(土)・7日(日),夜桜のライトアップ、和太鼓演奏、屋台,当日受付（先着順）,無料
春舞踊大会,京都市左京区,4月13日(土),伝統舞踊の発表、着物体験,事前申込（定員80名）,"2,000円（体験料込）"
こども和楽器教室,広島県福山市,4月21日(日),小学生対象の楽器体験、演奏会,事前申込（保護者同伴）,"1,000円（教材費含む）"
青空茶会,静岡県掛川市,4月27日(土),野外での茶道体験、抹茶サービス,当日受付（定員なし）,500円（抹茶代）


In [9]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N2_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N3 Level Exam

In [ ]:
runner = TaskRunner(level="N3", exam_type="fast_exam")
n3_outline, n3_exam_paper = runner.run()

2025-11-17 13:55:41,901 - INFO - jlpt - Module 'graphs.n3.outliner' imported successfully.
2025-11-17 13:55:41,901 - INFO - Module 'graphs.n3.outliner' imported successfully.
2025-11-17 13:55:41,903 - INFO - jlpt - Current Task ID None is being initialized
2025-11-17 13:55:41,903 - INFO - Current Task ID None is being initialized
2025-11-17 13:55:56,000 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-11-17 13:55:56,004 - INFO - jlpt - Outline of the exam:

# 日本語能力試験 N3 模擬試験

## 第1部：語彙

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい（1問）。80%が難易度の高い語彙から選定。

- **畜産**

### write_kanji

問題2：このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい（1問）。

- **しゅんかん**

### word_meaning

問題3：（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい（1問）。

- **ふまん**

### synonym_substitution

問題4：意味が最も近いものを、1・2・3・4から一つえらびなさい（1問）。

- **あわれ**

### word_usage

問題5：つぎのことばの使い方として最もよいものを、1・2・3・4から一

## N3 Outline Preview

In [ ]:
display(Markdown(n3_outline.as_str))

## N3 Exam Result

In [ ]:
html_output = render_to_html(n3_exam_paper['sections'])
display(HTML(html_output))

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./output/JLPT_N3_{timestamp}.html"

with open(filename, "w", encoding="utf-8") as file:
    file.write(html_output)

# N4 Level Exam

In [ ]:
# runner = TaskRunner(level="N4", exam_type="fast_exam")
# n4_outline, n4_exam_paper = runner.run()

## N4 Outline Preview

In [ ]:
# display(Markdown(n4_outline.as_str))

## N4 Exam Result

In [ ]:
# html_output = render_to_html(n4_exam_paper['sections'])
# display(HTML(html_output))

# N5 Level Exam

In [ ]:
# runner = TaskRunner(level="N5", exam_type="fast_exam")
# n5_outline, n5_exam_paper = runner.run()

## N5 Outline Preview

In [ ]:
# display(Markdown(n5_outline.as_str))

## N5  Exam Result

In [ ]:
# html_output = render_to_html(n5_exam_paper['sections'])
# display(HTML(html_output))